<h3> Imports and paths

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
print("PYTHONPATH:", PROJECT_ROOT)


PYTHONPATH: /Users/aidos/ML Projects Personal/Comp_BioChem_Project


In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

from src.baselines.features import FeatureConfig, build_features_for_chain, load_labels

ROOT = PROJECT_ROOT
META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"


<h3> Load splits</h3>

In [3]:
with open(META / "data_splits.json", "r") as f:
    splits = json.load(f)

print(len(splits["train"]), len(splits["val"]), len(splits["test"]))


180 35 35


In [4]:
def build_xy(split_name: str, t_angstrom: int = 5):
    X_list = []
    y_list = []
    feat_names = None

    cfg = FeatureConfig(include_flags=True, include_position=True)

    for ex in tqdm(splits[split_name], desc=f"features {split_name}"):
        ex_dir = PROCESSED / f'{ex["pdb_id"]}_{ex["chainA"]}_{ex["chainB"]}'

        for chain in ["A", "B"]:
            X, names = build_features_for_chain(ex_dir, chain, cfg=cfg)
            y = load_labels(ex_dir, chain, t_angstrom=t_angstrom)

            if feat_names is None:
                feat_names = names
            else:
                assert feat_names == names, "Feature name mismatch across examples!"

            X_list.append(X.astype(np.float32))
            y_list.append(y.astype(np.int8))

    X_all = np.vstack(X_list)
    y_all = np.concatenate(y_list)
    return X_all, y_all, feat_names

X_train, y_train, feat_names = build_xy("train", t_angstrom=5)
X_val,   y_val,   _          = build_xy("val",   t_angstrom=5)
X_test,  y_test,  _          = build_xy("test",  t_angstrom=5)

X_train.shape, y_train.mean(), X_val.shape, X_test.shape


features train:   0%|          | 0/180 [00:00<?, ?it/s]

features val:   0%|          | 0/35 [00:00<?, ?it/s]

features test:   0%|          | 0/35 [00:00<?, ?it/s]

((80934, 43), np.float64(0.11651469098277609), (17541, 43), (15798, 43))

<h3> Build X,y for a split

In [5]:
def build_xy(split_name: str, t_angstrom: int = 5):
    X_list = []
    y_list = []
    feat_names = None

    cfg = FeatureConfig(include_flags=True, include_position=True)

    for ex in tqdm(splits[split_name], desc=f"features {split_name}"):
        ex_dir = PROCESSED / f'{ex["pdb_id"]}_{ex["chainA"]}_{ex["chainB"]}'

        for chain in ["A", "B"]:
            X, names = build_features_for_chain(ex_dir, chain, cfg=cfg)
            y = load_labels(ex_dir, chain, t_angstrom=t_angstrom)

            if feat_names is None:
                feat_names = names
            else:
                assert feat_names == names, "Feature name mismatch across examples!"

            X_list.append(X.astype(np.float32))
            y_list.append(y.astype(np.int8))

    X_all = np.vstack(X_list)
    y_all = np.concatenate(y_list)
    return X_all, y_all, feat_names

X_train, y_train, feat_names = build_xy("train", t_angstrom=5)
X_val,   y_val,   _          = build_xy("val",   t_angstrom=5)
X_test,  y_test,  _          = build_xy("test",  t_angstrom=5)

X_train.shape, y_train.mean(), X_val.shape, X_test.shape


features train:   0%|          | 0/180 [00:00<?, ?it/s]

features val:   0%|          | 0/35 [00:00<?, ?it/s]

features test:   0%|          | 0/35 [00:00<?, ?it/s]

((80934, 43), np.float64(0.11651469098277609), (17541, 43), (15798, 43))

<h3> Train Logistic Regression

In [6]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="liblinear",
    max_iter=200,
    class_weight="balanced",
)

lr.fit(X_train, y_train)


LogisticRegression(class_weight='balanced', max_iter=200, solver='liblinear')

<h3> Metrics setup (PR-AUC, F1@K, P@K)

In [7]:
from sklearn.metrics import average_precision_score

def scores_to_topk(y_score: np.ndarray, k: int) -> np.ndarray:
    k = max(1, min(int(k), len(y_score)))
    idx = np.argsort(-y_score)[:k]
    pred = np.zeros(len(y_score), dtype=np.int8)
    pred[idx] = 1
    return pred

def f1_at_k(y_true, y_score, k):
    pred = scores_to_topk(y_score, k)
    tp = ((pred == 1) & (y_true == 1)).sum()
    fp = ((pred == 1) & (y_true == 0)).sum()
    fn = ((pred == 0) & (y_true == 1)).sum()
    prec = tp / max(1, tp + fp)
    rec  = tp / max(1, tp + fn)
    if prec + rec == 0:
        return 0.0
    return float(2 * prec * rec / (prec + rec))

def precision_at_k(y_true, y_score, k):
    pred = scores_to_topk(y_score, k)
    tp = ((pred == 1) & (y_true == 1)).sum()
    return float(tp / max(1, k))

def evaluate(y_true, y_score, k_list=(10,20,30)):
    out = {"prauc": float(average_precision_score(y_true, y_score))}
    for k in k_list:
        out[f"f1@{k}"] = f1_at_k(y_true, y_score, k)
        out[f"p@{k}"] = precision_at_k(y_true, y_score, k)
    return out


<h3> Evaluate Logistic Regression

In [8]:
val_score = lr.predict_proba(X_val)[:, 1]
test_score = lr.predict_proba(X_test)[:, 1]

lr_val = evaluate(y_val, val_score, k_list=(10,20,30))
lr_test = evaluate(y_test, test_score, k_list=(10,20,30))

pd.Series(lr_val), pd.Series(lr_test)


(prauc    0.340901
 f1@10    0.000000
 p@10     0.000000
 f1@20    0.000879
 p@20     0.100000
 f1@30    0.002632
 p@30     0.200000
 dtype: float64,
 prauc    0.334386
 f1@10    0.006531
 p@10     0.900000
 f1@20    0.012292
 p@20     0.850000
 f1@30    0.013689
 p@30     0.633333
 dtype: float64)

<h3> Feature importance (LogReg coefficients)

In [9]:
coef = lr.coef_.ravel()
imp = pd.DataFrame({"feature": feat_names, "coef": coef}).sort_values("coef", ascending=False)

print("Top positive (interface-favoring):")
display(imp.head(15))

print("Top negative (interface-disfavoring):")
display(imp.tail(15))


Top positive (interface-favoring):


,feature,coef
40,frac_aromatic_neighbors,3.927708
39,frac_polar_neighbors,0.428232
34,min_neighbor_dist,0.388886
42,relative_position,0.275587
4,aa_C,0.216051
26,is_sulfur,0.208226
1,aa_R,0.202250
14,aa_P,0.198037
28,is_pro,0.198037
38,frac_hydrophobic_neighbors,0.157682


Top negative (interface-disfavoring):


,feature,coef
31,net_charge,-0.033371
29,hydrophobicity_kd,-0.039953
0,aa_A,-0.039991
21,is_negative,-0.051686
41,mean_neighbor_degree,-0.071217
6,aa_E,-0.078189
20,is_positive,-0.085057
32,contact_degree,-0.107431
22,is_charged,-0.136743
10,aa_L,-0.260798


<br>
<br>

<h3> Random Forest

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=3,
    n_jobs=-1,
    class_weight="balanced_subsample",
    random_state=0
)

rf.fit(X_train, y_train)

rf_val_score = rf.predict_proba(X_val)[:, 1]
rf_test_score = rf.predict_proba(X_test)[:, 1]

rf_val = evaluate(y_val, rf_val_score, k_list=(10,20,30))
rf_test = evaluate(y_test, rf_test_score, k_list=(10,20,30))

pd.Series(rf_val), pd.Series(rf_test)


(prauc    0.389580
 f1@10    0.002644
 p@10     0.600000
 f1@20    0.005716
 p@20     0.650000
 f1@30    0.008335
 p@30     0.633333
 dtype: float64,
 prauc    0.359099
 f1@10    0.004354
 p@10     0.600000
 f1@20    0.008677
 p@20     0.600000
 f1@30    0.011527
 p@30     0.533333
 dtype: float64)

<h3> RF feature importance

In [11]:
rf_imp = pd.DataFrame({"feature": feat_names, "importance": rf.feature_importances_}).sort_values("importance", ascending=False)
display(rf_imp.head(20))


,feature,importance
41,mean_neighbor_degree,0.128723
32,contact_degree,0.109783
42,relative_position,0.095070
40,frac_aromatic_neighbors,0.092886
33,mean_neighbor_dist,0.079661
37,frac_charged_neighbors,0.064510
39,frac_polar_neighbors,0.063332
38,frac_hydrophobic_neighbors,0.059109
36,frac_negative_neighbors,0.056563
34,min_neighbor_dist,0.054216


<h3> Random Forest Grid Search</h3>

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
import pandas as pd
import numpy as np
from itertools import product

# keep these fixed for stability/reproducibility
BASE_RF_KW = dict(
    n_jobs=-1,
    class_weight="balanced_subsample",
    random_state=0,
)

grid = {
    "n_estimators": [300],          # keep fixed; can add 500 later if you want
    "max_depth": [12, 16, 20, None],
    "min_samples_leaf": [1, 3, 5],
}

def eval_scores(y_true, y_score, k_list=(10,20,30)):
    out = {"prauc": float(average_precision_score(y_true, y_score))}
    for k in k_list:
        out[f"f1@{k}"] = f1_at_k(y_true, y_score, k)
        out[f"p@{k}"] = precision_at_k(y_true, y_score, k)
    return out

rows = []
best = None

for n_estimators, max_depth, min_samples_leaf in product(
    grid["n_estimators"], grid["max_depth"], grid["min_samples_leaf"]
):
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        **BASE_RF_KW
    )
    rf.fit(X_train, y_train)

    val_score = rf.predict_proba(X_val)[:, 1]
    metrics = eval_scores(y_val, val_score, k_list=(10,20,30))

    row = {
        "n_estimators": n_estimators,
        "max_depth": -1 if max_depth is None else max_depth,
        "min_samples_leaf": min_samples_leaf,
        **metrics
    }
    rows.append(row)

    if best is None or row["prauc"] > best["prauc"]:
        best = row

df_grid = pd.DataFrame(rows).sort_values("prauc", ascending=False).reset_index(drop=True)
display(df_grid.head(10))
print("Best (by val PR-AUC):", best)


,n_estimators,max_depth,min_samples_leaf,prauc,f1@10,p@10,f1@20,p@20,f1@30,p@30
0,300,20,1,0.393082,0.003084,0.7,0.005276,0.60,0.008335,0.633333
1,300,-1,1,0.392469,0.002644,0.6,0.004836,0.55,0.005703,0.433333
2,300,-1,3,0.392231,0.003084,0.7,0.005276,0.60,0.008335,0.633333
3,300,20,3,0.391331,0.002644,0.6,0.006155,0.70,0.008774,0.666667
4,300,16,1,0.387637,0.003084,0.7,0.005716,0.65,0.007458,0.566667
5,300,16,3,0.387376,0.002644,0.6,0.005276,0.60,0.007896,0.600000
6,300,-1,5,0.386976,0.003084,0.7,0.005276,0.60,0.007896,0.600000
7,300,20,5,0.383915,0.002644,0.6,0.004836,0.55,0.008335,0.633333
8,300,16,5,0.383528,0.002203,0.5,0.005276,0.60,0.008335,0.633333
9,300,12,3,0.376485,0.002644,0.6,0.004836,0.55,0.007019,0.533333


Best (by val PR-AUC): {'n_estimators': 300, 'max_depth': 20, 'min_samples_leaf': 1, 'prauc': 0.39308165686083185, 'f1@10': 0.003084379819343468, 'p@10': 0.7, 'f1@20': 0.0052758848098483175, 'p@20': 0.6, 'f1@30': 0.008335161219565694, 'p@30': 0.6333333333333333}


<h3> Train final RF with best params and evaluate on test</h3>

In [13]:
best_max_depth = None if best["max_depth"] == -1 else int(best["max_depth"])

rf_best = RandomForestClassifier(
    n_estimators=int(best["n_estimators"]),
    max_depth=best_max_depth,
    min_samples_leaf=int(best["min_samples_leaf"]),
    **BASE_RF_KW
)
rf_best.fit(X_train, y_train)

val_score = rf_best.predict_proba(X_val)[:, 1]
test_score = rf_best.predict_proba(X_test)[:, 1]

rf_best_val = eval_scores(y_val, val_score, k_list=(10,20,30))
rf_best_test = eval_scores(y_test, test_score, k_list=(10,20,30))

print("RF best VAL:")
print(pd.Series(rf_best_val))
print("\nRF best TEST:")
print(pd.Series(rf_best_test))


RF best VAL:
prauc    0.393082
f1@10    0.003084
p@10     0.700000
f1@20    0.005276
p@20     0.600000
f1@30    0.008335
p@30     0.633333
dtype: float64

RF best TEST:
prauc    0.351707
f1@10    0.003628
p@10     0.500000
f1@20    0.007231
p@20     0.500000
f1@30    0.010807
p@30     0.500000
dtype: float64
